In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv('lending_club_sample.csv')
df.head()

In [ ]:
# 부도를 어떤 값으로 살펴볼 것인가?
# Charged off/Default: 부도(=1), Fully Paid: 부도 아님(=0)

df['default'] = df['loan_status'].apply(lambda x: 1 if x in ['Charged Off','Default'] else 0)

# 정보집합 관점에서의 사용 가능한 변수의 선택
features = ['loan_amnt','purpose','emp_length','annual_inc','dti','open_acc','revol_bal','revol_util','total_acc','delinq_2yrs','inq_last_6mths','fico_range_low']

# 최종 분석에 사용할 변수 추린 후, 약간의 전처리
df= df[features + ['default','int_rate','term','verification_status']].dropna()
df['loan_duration_years']=df['term'].apply(lambda x: 3 if '36' in x else 5)

In [ ]:
# 원본 데이터를 보존하여 나중에 활용하기 위함.
original_data = df.copy()

# categorical data의 더미변수 처리
df = pd.get_dummies(df, columns = ['purpose','emp_length','verification_status'], drop_first=True)
df = pd.get_dummies(df, columns = ['term'], drop_first=True)

In [ ]:
# Train-test-split:

X = df.drop(columns=['default'])
y = df['default']

X_temp, X_test, y_temp, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp, test_size=0.25, random_state=42)

original_temp, original_test = train_test_split(original_data, test_size=0.2, random_state=42)
original_train, original_val = train_test_split(original_temp, test_size=0.25, random_state=42)

In [ ]:
# 변수의 표준화 (LASSO 사용 시, 변수의 scale을 맞춰주기 위함.)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
continuous_features = ['loan_amnt','annual_inc','dti','open_acc','revol_bal','revol_util','total_acc','delinq_2yrs','inq_last_6mths']

X_train[continuous_features]=scaler.fit_transform(X_train[continuous_features])
X_val[continuous_features]=scaler.transform(X_val[continuous_features])


In [ ]:
## Train set의 balance
from sklearn.utils import resample

X_train_balanced = pd.concat([X_train, y_train],axis=1)
default_data = X_train_balanced[X_train_balanced['default']==1]
non_default_data = X_train_balanced[X_train_balanced['default']==0]

non_default_downsampled =  resample(
    non_default_data,
    replace=False,
    n_samples = len(default_data),
    random_state=42)

balanced_train_data = pd.concat([default_data, non_default_downsampled])

X_train = balanced_train_data.drop(columns=['default'])
y_train = balanced_train_data['default']

지금부터 threshold에 따라서

1) 먼저 예측치를 구하고
2) 하나의 threshold에 따라서
3) 우리의 목적함수를 설정 $s = \frac{\mu_r - r_f}{\sigma_{r}}$
4) threshold에 따라 분류된 결과에 따라 sharpe ratio가 어떻게 구해는지 판단해보기

In [ ]:
from sklearn.linear_model import Lasso

alphas = [0.01, 0.1, 1]
thresholds = [0.2, 0.3, 0.4]
risk_free_rate = 0.03

best_sharpe = -np.inf
best_alpha = None
best_threshold = None

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train, y_train)
    y_pred_val = lasso.predict(X_val)

    for threshold in thresholds:
        predictions = (y_pred_val >= threshold).astype(int)

        approved_loans_val = (predictions == 0)
        not_approved_loans_val = (predictions == 1)

        individual_annualized_returns_val = (
            approved_loans_val * (y_val == 0) * (
                ((original_val['loan_amnt'] * ((1 + original_val['int_rate'] / 100) ** original_val['term'].apply(lambda x: 3 if '36' in x else 5))) / original_val['loan_amnt']) ** (1 / original_val['term'].apply(lambda x: 3 if '36' in x else 5)) - 1
            ) +
            not_approved_loans_val * (
                ((original_val['loan_amnt'] * ((1 + risk_free_rate) ** original_val['term'].apply(lambda x: 3 if '36' in x else 5))) / original_val['loan_amnt']) ** (1 / original_val['term'].apply(lambda x: 3 if '36' in x else 5)) - 1
            )
        ).dropna()

        mean_return = individual_annualized_returns_val.mean()
        std_return = individual_annualized_returns_val.std()
        sharpe = (mean_return - risk_free_rate) / std_return if std_return > 0 else -np.inf

        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_alpha = alpha
            best_threshold = threshold

print(f"Best alpha: {best_alpha}")
print(f"Best threshold: {best_threshold}")
print(f"Best Sharpe Ratio: {best_sharpe:.4f}")


In [ ]:
X_temp[continuous_features]=scaler.fit_transform(X_temp[continuous_features])
X_test[continuous_features]=scaler.transform(X_test[continuous_features])

In [ ]:
alpha= 0.01
threshold= 0.4

# 모델 학습
model = Lasso(alpha=alpha)
model.fit(X_temp, y_temp)

# 예측 (연속값)
y_pred = model.predict(X_test)

# Threshold 기준으로 이진화

binary_predictions = (y_pred >= threshold).astype(int)

In [ ]:

feature_names = X_temp.columns


lasso_coefficients = pd.Series(model.coef_, index=feature_names)


print(lasso_coefficients.sort_values(ascending=False))

In [ ]:

approved = (binary_predictions == 0)
not_approved = (binary_predictions == 1)


years = original_test['term'].apply(lambda x: 3 if '36' in x else 5)


approved_returns = approved * (y_test == 0) * (
    ((original_test['loan_amnt'] * ((1 + original_test['int_rate'] / 100) ** years)) / original_test['loan_amnt']) ** (1 / years) - 1
)


not_approved_returns = not_approved * (
    ((original_test['loan_amnt'] * ((1 + risk_free_rate) ** years)) / original_test['loan_amnt']) ** (1 / years) - 1
)


individual_returns = (approved_returns + not_approved_returns).dropna()


mean_return = individual_returns.mean()
std_return = individual_returns.std()
risk_free_rate = 0.03

sharpe_ratio = (mean_return - risk_free_rate) / std_return

print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
